# Fraud Detection — Graph Intelligence Demo

End-to-end walkthrough:
1. Generate synthetic transactions (normal + fraud)
2. Build the transaction graph
3. Compute structural, temporal, and behavioural features
4. Score all accounts with the weighted + anomaly model
5. Detect suspicious communities / mule rings
6. Print explainability reports

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import warnings
warnings.filterwarnings('ignore')

## 1. Generate Synthetic Data

In [ ]:
from src.simulation.normal_patterns import generate_mixed_normal
from src.simulation.fraud_patterns import generate_mixed_fraud
import time

BASE_TS = time.time() - 7200

normal_txns = generate_mixed_normal(total=500, seed=42)
fraud_txns  = generate_mixed_fraud(seed=99, start_ts=BASE_TS)

all_txns = normal_txns + fraud_txns
print(f'Normal: {len(normal_txns)}  |  Fraud: {len(fraud_txns)}  |  Total: {len(all_txns)}')

## 2. Build the Transaction Graph

In [ ]:
from src.graph.graph_builder import build_graph, graph_summary

G = build_graph(all_txns)
stats = graph_summary(G)
print('Graph stats:', stats)

## 3. Feature Engineering (sample node)

In [ ]:
from src.features.structural import get_structural_features
from src.features.temporal   import get_temporal_features
from src.features.behavioral import get_behavioral_features
import pandas as pd

sample_node = list(G.nodes())[0]
feats = {}
feats.update(get_structural_features(G, sample_node))
feats.update(get_temporal_features(G, sample_node))
feats.update(get_behavioral_features(G, sample_node))

print(f'Features for node: {sample_node}')
pd.Series(feats).round(4)

## 4. Score All Accounts

In [ ]:
from src.scoring.weighted_scorer import WeightedScorer
from src.scoring.anomaly_model   import AnomalyModel
from src.scoring.normalizer      import build_risk_result

scorer = WeightedScorer()
amodel = AnomalyModel()

# Compute feature vectors for all nodes
all_features = {}
for node in G.nodes():
    f = {}
    f.update(get_structural_features(G, node))
    f.update(get_temporal_features(G, node))
    f.update(get_behavioral_features(G, node))
    all_features[node] = f

# Train anomaly model
amodel.train(list(all_features.values()))
print('Anomaly model trained.')

# Score every node
results = {}
for node, feats in all_features.items():
    out   = scorer.score(feats)
    boost = amodel.anomaly_boost(feats)
    results[node] = build_risk_result(node, out['weighted_score'], boost,
                                      out['contributions'], out['missing'])

scores_df = pd.DataFrame([
    {'account': k, 'risk_score': v['risk_score'], 'tier': v['tier']}
    for k, v in results.items()
]).sort_values('risk_score', ascending=False)

print(f"\nTop 10 highest-risk accounts:")
scores_df.head(10)

## 5. Score Distribution

In [ ]:
import matplotlib.pyplot as plt

scores_df['risk_score'].hist(bins=30, color='steelblue', edgecolor='white')
plt.axvline(0.60, color='orange', linestyle='--', label='High threshold')
plt.axvline(0.85, color='red',    linestyle='--', label='Critical threshold')
plt.xlabel('Risk Score')
plt.ylabel('Account Count')
plt.title('Risk Score Distribution')
plt.legend()
plt.tight_layout()
plt.show()

print(scores_df['tier'].value_counts())

## 6. Community / Cluster Detection

In [ ]:
from src.network.community_detection import detect_communities
from src.network.cluster_scorer      import score_all_clusters

communities = detect_communities(G)
print(f'Communities detected: {len(communities)}')

node_scores = {n: r['risk_score'] for n, r in results.items()}
clusters = score_all_clusters(G, communities, node_scores, min_members=3)

mule_rings = [c for c in clusters if c['is_mule_ring']]
print(f'Mule rings: {len(mule_rings)}')
print(f'High-risk clusters (≥ 0.6): {sum(1 for c in clusters if c["cluster_risk_score"] >= 0.6)}')

if clusters:
    top = clusters[0]
    print(f'\nTop cluster: {len(top["members"])} members, risk={top["cluster_risk_score"]:.2f}, mule={top["is_mule_ring"]}')

## 7. Explainability Reports

In [ ]:
from src.explainability.report_generator import generate_report

cluster_map = {}
for c in clusters:
    for m in c['members']:
        cluster_map[m] = c

# Print reports for top 3 riskiest accounts
for _, row in scores_df.head(3).iterrows():
    node = row['account']
    report = generate_report(
        node,
        results[node],
        cluster_info=cluster_map.get(node),
        features=all_features[node],
    )
    print(report)

## 8. Verify: Fraud vs Normal Score Separation

In [ ]:
# Build a label map from the raw transactions
label_map = {}
for t in all_txns:
    label_map[t['from_account']] = t.get('label', 'unknown')
    label_map[t['to_account']]   = t.get('label', 'unknown')

scores_df['label'] = scores_df['account'].map(label_map)

print('Mean risk by label:')
print(scores_df.groupby('label')['risk_score'].agg(['mean','median','count']).round(3))

scores_df.boxplot(column='risk_score', by='label', figsize=(6,4))
plt.suptitle('')
plt.title('Risk Score by Label')
plt.xlabel('Label')
plt.ylabel('Risk Score')
plt.tight_layout()
plt.show()